In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report
import joblib, os

matches    = pd.read_csv('/backend/data/all_matches.csv')
deliveries = pd.read_csv('/backend/data/all_deliveries.csv', low_memory=False)
os.makedirs('../backend/model', exist_ok=True)
print("Libraries and data loaded!!!")

Libraries and data loaded!!!


In [7]:
# Filter clean matches (no D/L, no no-result, no super over)
clean = matches[
    matches['winner'].notna() &
    (matches['method'].isna() | (matches['method'] == '')) &
    (matches['super_over'] == 'N')
].copy().reset_index(drop=True)

# Encode teams and city
le_team1 = LabelEncoder()
le_city1 = LabelEncoder()
le_team1.fit(pd.concat([clean['team1'], clean['team2']]))

clean['team1_enc']  = le_team1.transform(clean['team1'])
clean['team2_enc']  = le_team1.transform(clean['team2'])
clean['city']       = clean['city'].fillna('Unknown')
clean['city_enc']   = le_city1.fit_transform(clean['city'])
clean['toss_team1'] = (clean['toss_winner'] == clean['team1']).astype(int)
clean['toss_bat']   = (clean['toss_decision'] == 'bat').astype(int)
clean['target']     = (clean['winner'] == clean['team1']).astype(int)

# Add historical win rate per team
all_teams_list = pd.concat([clean['team1'], clean['team2']]).unique()
win_rates = {}
for team in all_teams_list:
    played = clean[(clean['team1']==team) | (clean['team2']==team)]
    wins   = clean[clean['winner']==team]
    win_rates[team] = round(len(wins)/len(played), 4) if len(played) > 0 else 0.5

clean['team1_win_rate'] = clean['team1'].map(win_rates)
clean['team2_win_rate'] = clean['team2'].map(win_rates)

# Head-to-head win rate
def get_h2h(df):
    result = []
    for _, row in df.iterrows():
        mask = (
            ((df['team1']==row['team1']) & (df['team2']==row['team2'])) |
            ((df['team1']==row['team2']) & (df['team2']==row['team1']))
        )
        h2h = df[mask]
        t1w = h2h[h2h['winner']==row['team1']]
        result.append(round(len(t1w)/len(h2h), 4) if len(h2h)>0 else 0.5)
    return result

clean['h2h_rate'] = get_h2h(clean)

# Train / Test split (season-based — no data leakage)
f1 = ['team1_enc','team2_enc','city_enc','toss_team1','toss_bat',
      'team1_win_rate','team2_win_rate','h2h_rate']
train1 = clean[clean['season'] <= 2022]
test1  = clean[clean['season'] >= 2023]

m1 = LogisticRegression(max_iter=1000)
m1.fit(train1[f1], train1['target'])

acc = accuracy_score(test1['target'], m1.predict(test1[f1]))
print(f"✅ Model 1 Accuracy: {acc*100:.1f}%")

joblib.dump({
    'model': m1, 'le_team': le_team1, 'le_city': le_city1,
    'features': f1, 'win_rates': win_rates,
    'team_list': sorted(le_team1.classes_.tolist()),
    'city_list': sorted(le_city1.classes_.tolist())
}, '../backend/model/match_winner.pkl')
print("✅ match_winner.pkl saved")

✅ Model 1 Accuracy: 56.3%
✅ match_winner.pkl saved


In [8]:
inn = deliveries[deliveries['inning']==1].groupby('match_id').agg(
    total_runs  =('total_runs','sum'),
    batting_team=('batting_team','first'),
    bowling_team=('bowling_team','first')
).reset_index()

inn = inn.merge(
    matches[['id','season','city','toss_decision','toss_winner','method','winner']],
    left_on='match_id', right_on='id'
)
inn = inn[inn['winner'].notna() & (inn['method'].isna() | (inn['method']==''))]

le_team2 = LabelEncoder()
le_city2 = LabelEncoder()
le_team2.fit(pd.concat([inn['batting_team'], inn['bowling_team']]))

inn['batting_enc']  = le_team2.transform(inn['batting_team'])
inn['bowling_enc']  = le_team2.transform(inn['bowling_team'])
inn['city']         = inn['city'].fillna('Unknown')
inn['city_enc']     = le_city2.fit_transform(inn['city'])
inn['toss_bat']     = (inn['toss_decision']=='bat').astype(int)
inn['batting_toss'] = (inn['toss_winner']==inn['batting_team']).astype(int)

f2 = ['batting_enc','bowling_enc','city_enc','season','toss_bat','batting_toss']
train2 = inn[inn['season']<=2022]
test2  = inn[inn['season']>=2023]

m2 = RandomForestRegressor(n_estimators=100, random_state=42)
m2.fit(train2[f2], train2['total_runs'])

mae = mean_absolute_error(test2['total_runs'], m2.predict(test2[f2]))
print(f"✅ Model 2 MAE: ±{mae:.1f} runs")

joblib.dump({
    'model': m2, 'le_team': le_team2, 'le_city': le_city2,
    'features': f2,
    'team_list': sorted(le_team2.classes_.tolist()),
    'city_list': sorted(le_city2.classes_.tolist())
}, '../backend/model/innings_score.pkl')
print("✅ innings_score.pkl saved")

✅ Model 2 MAE: ±32.2 runs
✅ innings_score.pkl saved


In [ ]:
mport pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
import joblib
import psycopg2 

# 1. Fetch data
conn = psycopg2.connect("dbname=ipl_dashboard user=postgres password=comf@psql host=localhost")
df = pd.read_sql_query("SELECT * FROM player_season_stats", conn)
conn.close()

# 2. Feature Engineering: Composite Impact Score
# Optional: Add catches if you have them! e.g., + (df['catches'] * 8)
df['impact_score'] = df['total_runs'] + (df['wickets'] * 25) 

# 3. Create Labels
q33 = df['impact_score'].quantile(0.33)
q67 = df['impact_score'].quantile(0.67)

df['label'] = pd.cut(df['impact_score'],
    bins=[-1, q33, q67, float('inf')],
    labels=['Poor', 'Average', 'Good'])

df = df.dropna(subset=['label'])

# 4. Define features
features = ['total_runs', 'strike_rate', 'batting_avg', 'wickets', 'economy', 'bowling_avg']

# 🚨 THE FIX: Separate fillna logic for batting vs bowling
# Non-batters get 0s
batting_cols = ['total_runs', 'strike_rate', 'batting_avg']
df[batting_cols] = df[batting_cols].fillna(0)

# Non-bowlers get -1 so the Decision Tree knows they didn't bowl
bowling_cols = ['wickets', 'economy', 'bowling_avg']
df[bowling_cols] = df[bowling_cols].fillna(-1)

# 5. Train / Test Split
train = df[df['season'] <= 2022]
test = df[df['season'] >= 2023]

# 6. Train the Model
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(train[features], train['label'])

print("--- Model Accuracy ---")
print(classification_report(test['label'], model.predict(test[features])))

# 7. Save the new model
joblib.dump({
    'model': model, 
    'features': features,
    'thresholds': {'q33': round(q33, 2), 'q67': round(q67, 2)}
}, 'player_performance.pkl')

print("✅ NEW player_performance.pkl saved!")

              precision    recall  f1-score   support

     Average       1.00      1.00      1.00       139
        Good       1.00      1.00      1.00       204
        Poor       1.00      1.00      1.00       173

    accuracy                           1.00       516
   macro avg       1.00      1.00      1.00       516
weighted avg       1.00      1.00      1.00       516


Decision Rules Learned:
|--- avg_per_inning <= 6.54
|   |--- class: Poor
|--- avg_per_inning >  6.54
|   |--- avg_per_inning <= 18.32
|   |   |--- class: Average
|   |--- avg_per_inning >  18.32
|   |   |--- class: Good

✅ player_performance.pkl saved


In [10]:
import os
print(os.listdir('../backend/model/'))
# ['match_winner.pkl', 'innings_score.pkl', 'player_performance.pkl']

['match_winner.pkl', 'innings_score.pkl', 'player_performance.pkl']


In [19]:
# Cell 1 — run this once to load everything
exec(open('test_models.py').read())


FileNotFoundError: [Errno 2] No such file or directory: 'test_models.py'